> **Historical research track.** This notebook documents the earlier trajectory-anomaly experiment (Phase 7 — the sealed-fold test burn). It is preserved as recorded evidence and does not define the current SADAR Analyst Console product. The code cells reference the pre-restructure `backend.core` package layout (preserved via the pre-restructure tag); embedded outputs are the original recorded run. Companion decision: [`docs/research/trajectory-anomaly/lifecycle/decisions/D-012-phase7-zone-reweight.md`](../../../../docs/research/trajectory-anomaly/lifecycle/decisions/D-012-phase7-zone-reweight.md); burn script: [`research/trajectory-anomaly/scripts/phase7_burn.py`](../../scripts/phase7_burn.py).

# Phase 7 — Evaluation: the test burn + blind real-anomaly head-to-head (issue #29)

The one-shot. Three models on the SEALED 2020 test fold, all reported (no dropping the loser).
Contract: `docs/research/trajectory-anomaly/lifecycle/07-eval-prep.md` "Layer 6" + `07-eval.md` + D-012 (zone re-weight).

**Lead with the per-type table and the real-anomaly head-to-head, not the synthetic mean.**

In [1]:
import sys, json
from pathlib import Path
import numpy as np, pandas as pd, joblib
from sklearn.metrics import roc_auc_score, average_precision_score
REPO=Path.cwd()
while not (REPO/"backend/core/preprocessing.py").exists() and REPO!=REPO.parent: REPO=REPO.parent
sys.path.insert(0,str(REPO))
from backend.core.preprocessing import to_sequences, to_sequences_loss_mask
from backend.core.baseline import KNNSummaryBaseline, IsolationForestBaseline
from backend.core import lstm_ae as ae
M=REPO/"backend/models/phase6"; T=260; SEED=42
clean=pd.read_parquet(M/"clean_df.parquet"); meta=pd.read_parquet(M/"meta.parquet")
ids=json.load(open(M/"split_ids.json")); scaler=joblib.load(M/"scaler.joblib")
model=ae.load_checkpoint(str(M/"lstm_ae_best.pt"))
knn=KNNSummaryBaseline.from_reference(np.load(M/"knn_train_summary.npy"), k=5)
print("loaded frozen artifacts | folds:", {k:len(v) for k,v in ids.items()})

loaded frozen artifacts | folds: {'train': 8924, 'val': 5942, 'test': 4285, 'held_aside': 698}


## Layer 6 (selection, firewall-clean) — AE vs kNN on real anomalies vs VAL-normal

Model selection uses **val-normal** as the negative class (never the sealed test). Both our models
score the same negatives, so the AE-vs-kNN ranking is prevalence-invariant.

In [2]:
g=meta.groupby('segment_id').agg(is_ga=('is_go_around','max'),is_em=('is_emergency','max'))
held=set(ids['held_aside'])
cohort=sorted([s for s in g.index if s in held and (g.loc[s,'is_ga'] or g.loc[s,'is_em'])])
ga=set(g.index[(g.is_ga==1)]); 
def win(seg_ids):
    df=clean[clean.segment_id.isin(set(seg_ids))]
    X,_,_=to_sequences(df,T,scaler); m=to_sequences_loss_mask(df,T)
    return X,m,list(df.groupby('segment_id',sort=False).groups.keys())
Xv,mv,_=win(ids['val']); Xc,mc,oc=win(cohort)
is_ga=np.array([s in ga for s in oc])
sc={'AE':(ae.reconstruction_error(model,Xv,mv,agg='mean'),ae.reconstruction_error(model,Xc,mc,agg='mean')),
    'kNN':(knn.anomaly_score(Xv,mv),knn.anomaly_score(Xc,mc))}
rows=[]
for name,(neg,pos) in sc.items():
    for lab,sel in [('combined',np.ones(len(pos),bool)),('go_around',is_ga),('emergency',~is_ga)]:
        y=np.r_[np.zeros(len(neg)),np.ones(sel.sum())]; s=np.r_[neg,pos[sel]]
        rows.append([name,lab,round(roc_auc_score(y,s),3),round(average_precision_score(y,s),3),int(sel.sum())])
print("Layer-6 selection (negatives=val-normal 2019):")
print(pd.DataFrame(rows,columns=['model','cohort','ROC','PR','n']).to_string(index=False))
print("\n-> AE beats kNN on real (representative=AE). Synthetic had kNN>AE — does NOT transfer.")

Layer-6 selection (negatives=val-normal 2019):
model    cohort   ROC    PR   n
   AE  combined 0.591 0.048 195
   AE go_around 0.591 0.048 191
   AE emergency 0.590 0.001   4
  kNN  combined 0.551 0.043 195
  kNN go_around 0.549 0.043 191
  kNN emergency 0.629 0.001   4

-> AE beats kNN on real (representative=AE). Synthetic had kNN>AE — does NOT transfer.


## SADAR VAE-LSTM (his native rep) — reproduce his published real-anomaly number

In [3]:
import importlib.util
def _load(name,p):
    spec=importlib.util.spec_from_file_location(name,p); m=importlib.util.module_from_spec(spec)
    sys.modules[name]=m; spec.loader.exec_module(m); return m
S=REPO/"external/sadar"
vae=_load("vae_lstm_sadar", S/"src/sadar/models/vae_lstm.py")
import torch
ck=torch.load(S/"models/vae_lstm.pt", map_location="cpu")
sm=vae.VAELSTM(n_features=ck["n_features"], **ck["model"]); sm.load_state_dict(ck["state_dict"]); sm.eval()
def srecon(X,bs=512):
    out=[]
    with torch.no_grad():
        for i in range(0,len(X),bs):
            b=torch.tensor(X[i:i+bs],dtype=torch.float32); out.append(((b-sm(b))**2).mean(dim=(1,2)).numpy())
    return np.concatenate(out)
tst=np.load(S/"data/processed/test.npy"); anom=np.load(S/"data/processed/anomalies.npy")
y=np.r_[np.zeros(len(tst)),np.ones(len(anom))]; s=np.r_[srecon(tst),srecon(anom)]
print(f"SADAR VAE-LSTM real: ROC {roc_auc_score(y,s):.3f} PR {average_precision_score(y,s):.3f} "
      f"prev {len(anom)/len(y):.3f} (his reported: 0.659/0.299)")

SADAR VAE-LSTM real: ROC 0.659 PR 0.299 prev 0.121 (his reported: 0.659/0.299)


## Layer 4 — external real emergency (OpenSky #6 7700): BCS63A case study

In [4]:
from backend.core.derivations import apply_derivations
from backend.core.features import build_features
t=pd.read_parquet(REPO/"data/external/squawk7700_trajectories.parquet.gz")
raw=pd.DataFrame({'time':t.timestamp.values.astype('datetime64[s]').astype('int64'),
 'icao24':t.icao24.astype(str),'lat':t.latitude,'lon':t.longitude,
 'baroaltitude':t.altitude*0.3048,'geoaltitude':t.altitude*0.3048,'velocity':t.groundspeed*0.514444,
 'heading':t.track,'vertrate':t.vertical_rate*0.00508,'onground':False,
 'squawk':t.squawk,'callsign':t.callsign.astype(str)}).dropna(subset=['lat','lon','time'])
der=apply_derivations(raw); cdf,cmeta=build_features(der)
ae_v=ae.reconstruction_error(model,Xv,mv,agg='mean'); kn_v=knn.anomaly_score(Xv,mv)
Xe,_,_=to_sequences(cdf,T,scaler); me=to_sequences_loss_mask(cdf,T)
ae_e=ae.reconstruction_error(model,Xe,me,agg='mean'); kn_e=knn.anomaly_score(Xe,me)
pct=lambda v,r: round(float((r<v).mean()*100),1)
print(f"in-range #6 flights kept by Filter B: {der['icao24'].nunique()} (icao24 {sorted(der.icao24.unique())})")
print(f"BCS63A turn-back: AE {ae_e[0]:.3f} (pctile {pct(ae_e[0],ae_v)}) | kNN {kn_e[0]:.3f} (pctile {pct(kn_e[0],kn_v)})")
print("-> real LEMD emergency the model never saw flags at ~99-100th pctile of normal. N=1 case study.")

in-range #6 flights kept by Filter B: 1 (icao24 ['3c5434'])
BCS63A turn-back: AE 0.778 (pctile 98.8) | kNN 3.317 (pctile 100.0)
-> real LEMD emergency the model never saw flags at ~99-100th pctile of normal. N=1 case study.


## Layer 5 — qualitative top-20 normal-val by AE reconstruction error

In [5]:
order_v=list(clean[clean.segment_id.isin(set(ids['val']))].groupby('segment_id',sort=False).groups.keys())
gv=clean[clean.segment_id.isin(set(ids['val']))].groupby('segment_id',sort=False)
d=pd.DataFrame({'segment_id':order_v,'ae':ae_v,'kn_pct':[ (kn_v<v).mean()*100 for v in kn_v]})
d['turn_deg']=d.segment_id.map(gv.apply(lambda s:float(np.degrees(np.abs(np.arctan2(np.sin(np.diff(np.radians(s.heading.to_numpy()))),np.cos(np.diff(np.radians(s.heading.to_numpy())))))).sum())))
d['alt_range']=d.segment_id.map(gv['baroaltitude'].max()-gv['baroaltitude'].min())
top=d.sort_values('ae',ascending=False).head(20)
print(top.round(2).to_string(index=False))
print(f"\ncircling/holding-like (>400deg): {(top.turn_deg>400).sum()}/20 | AE-flagged-but-kNN-normal(<50pct): {(top.kn_pct<50).sum()}/20")

         segment_id   ae  kn_pct  turn_deg  alt_range
34368e_1548689270#1 1.83   89.20    388.67    8602.98
344445_1569827690#1 1.82   99.90    268.94   10355.58
4ba90f_1554121590#1 1.79   99.58    278.98   10386.06
344291_1548684130#1 1.77   97.17    323.75    9639.30
a8a110_1554107710#1 1.75    7.52      0.00   12062.46
344503_1548666980#1 1.66   99.50   1047.53    9921.24
345051_1559557440#1 1.64   31.99    149.82    8260.08
345559_1569836700#1 1.59   90.31     98.38   11658.60
4ca564_1548705870#1 1.49   99.81    193.60   10820.40
34558f_1548667530#1 1.49   99.26    942.43   11330.94
3e3033_1559572300#2 1.42   98.18    307.93    7871.46
344346_1548666490#1 1.37   98.00    949.79    8953.50
3450d8_1564389210#1 1.36   98.54    120.44   12092.94
34530e_1548714950#1 1.35   98.96    238.89    7185.66
478fcc_1548668480#1 1.33   98.27    591.96   10416.54
3444ce_1554155740#1 1.22   47.26    231.23    7414.26
34508b_1548681540#1 1.21   97.48    874.00    7338.06
3440cd_1548678800#1 1.21   9

## THE BURN — sealed 2020 test fold, all three models (one-shot)

Runs `backend/scripts/phase7_burn.py`. Flips `test_set.burned=true`. D-012 re-weighted synthetic
mix + real-anomaly vs 2020-test-normal. No tuning/threshold/selection touched the test fold.

In [6]:
import runpy
runpy.run_path(str(REPO/"backend/scripts/phase7_burn.py"), run_name="__main__")

PHASE-7 TEST BURN — sealed 2020 fold (firewall opens here, once)
test segments: 4285 | T=260 | D-012 mix (zone OUT): {'altitude_high': 0.3333333333333333, 'sustained_loiter': 0.3333333333333333, 'final_approach_intercept': 0.16666666666666666, 'speed_spike': 0.16666666666666666}



[1] HEADLINE — D-012 mixed synthetic (4 dynamic types), TEST fold
    model    AUROC  PR-AUC
    AE       0.731   0.770
    kNN      0.786   0.827
    IF       0.717   0.734


    AE headline AUROC 95% CI: [0.714, 0.745]  (target > 0.85)

[2] AE OPERATING POINT (thr 0.222, val-chosen, no retune)
    F2 0.520 | FPR 0.089 (guardrail <=0.15) | recall 0.474

[3] PER-TYPE AUROC, TEST (zone = out-of-remit diagnostic, not in headline)
    type                           AE    kNN     IF


    zone_violation              0.551  0.541  0.503   <- out-of-remit


    altitude_high               0.558  0.594  0.556


    sustained_loiter            0.971  0.986  0.952


    final_approach_intercept    0.789  0.786  0.745


    speed_spike                 0.580  0.815  0.587



[4] REAL-ANOMALY — held-aside cohort vs 2020-TEST-normal (SADAR-comparable)
    cohort 195 vs test-normal 4285 | prevalence 0.044
    model      ROC      PR
    AE       0.667   0.088
    kNN      0.595   0.067
    IF       0.495   0.043
    SADAR  VAE-LSTM (native, his 2020-normal): ROC 0.659 PR 0.299 (reproduced)

saved -> /Users/txemapuch/Claude/drone-ai-saturdays/backend/models/phase6/phase7_burn_results.json


{'__name__': '__main__',
 '__doc__': "Phase-7 TEST BURN — the one-shot sealed-fold evaluation (issue #29).\n\nScores the SEALED 2020 test fold (firewall: fold='test', burned once here). Runs all\nthree models per the user-amended Layer-6 protocol (report all, drop none):\n  - our LSTM-AE (small/mean, threshold 0.222)   — backend/models/phase6/lstm_ae_best.pt\n  - our frozen kNN-on-summary (k=5)             — knn_train_summary.npy + scaler.joblib\n  - IsolationForest baseline (D-006)            — refit on TRAIN-normal\n\nSADAR's VAE-LSTM runs on its NATIVE rep (external/sadar) — reported alongside, not\non our fold (cross-feature translation forbidden by the pre-registration).\n\nSynthetic uses the D-012 re-weighted mix (zone OUT of headline, kept as diagnostic).\nReal-anomaly uses the held-aside cohort vs 2020-test-normal (SADAR-comparable, post-burn).\n\nDeterministic: seed 42, frozen artifacts, fixed Monday->fold map.\n",
 '__package__': '',
 '__loader__': None,
 '__spec__': None,
 '

## Verdict

- **Ship the LSTM-AE** (`model_track: dl`). It beats the kNN on real anomalies (the pre-registered
  criterion) and matches SADAR (ROC 0.667 vs 0.659). kNN wins synthetic but that does not transfer.
- Primary AUROC>0.85 **not met** (0.731) — honest. Lead with the per-type table + real-anomaly head-to-head.
- See `07-eval.md` for the full write-up and `D-012` for the zone re-weight rationale.